*Adapted from code shared by David Gorkin (shared 9/29/25)*

**Format GTEx expression metadata (TPMs from adult tissues), for use with tau index calculation**

# Settings

In [1]:
library(dplyr)
library(gplots)
library(RColorBrewer)
library(here)
here::i_am("figures/Figure2/tau/format_expression_data_gtex.ipynb")
sessionInfo()


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


here() starts at /mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics/figures/Figure2/tau

here() starts at /mnt/disk/home/agulka/manuscripts/BAF_stimulus_CellGenomics



R version 4.2.2 Patched (2022-11-10 r83330)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 23.04

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] here_1.0.2         RColorBrewer_1.1-3 gplots_3.2.0       dplyr_1.1.4       

loaded via a namespace (and not attached):
 [1] magrittr_2.0.3     tidyselect_1.2.1   uuid_1.2-1         R6_2.5.1          
 [5] rlang_1.1.4        fastmap_1.2.0      fansi_1.0.6        caTools_1.18.3    
 [9] too

In [2]:
# Input files

# - rowdata gene table from GENCODE
rowdata.file <- here(
  "inputs", "GENCODE",
  "v47_rowdata_human.rds"
)

# - rowdata column to use for matching
match.col <- "gene_id_base"

# - unformatted expression data
expr.file <- here(
  "inputs", "GTEx",
  "GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm.gct.gz"
)

# Output files
out.files <- list(
  coldata = here("inputs", "GTEx", "coldata.rds"),
  exprmat = here("inputs", "GTEx", "exprmat.rds"),
  layout  = here("inputs", "GTEx", "layout.rds")
)


# Load input data

In [3]:
rowdata <- readRDS(rowdata.file)
nrow(rowdata)
head(rowdata)

[1] 30696

,source,type,gene_id,gene_type,gene_name,level,tag,hgnc_id,havana_gene,artif_dupl,gene_id_base,chr,start,end
,<fct>,<fct>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
ENSG00000310526.1,HAVANA,gene,ENSG00000310526.1,lncRNA,WASH7P,2,overlaps_pseudogene,HGNC:38034,NA,NA,ENSG00000310526,chr1,14356,30745
ENSG00000278267.1,ENSEMBL,gene,ENSG00000278267.1,miRNA,MIR6859-1,3,NA,HGNC:50039,NA,NA,ENSG00000278267,chr1,17369,17437
ENSG00000243485.6,HAVANA,gene,ENSG00000243485.6,lncRNA,MIR1302-2HG,2,ncRNA_host,HGNC:52482,OTTHUMG00000000959.2,NA,ENSG00000243485,chr1,28589,31110
ENSG00000284332.1,ENSEMBL,gene,ENSG00000284332.1,miRNA,MIR1302-2,3,NA,HGNC:35294,NA,NA,ENSG00000284332,chr1,30366,30504
ENSG00000237613.3,HAVANA,gene,ENSG00000237613.3,lncRNA,FAM138A,2,NA,HGNC:32334,OTTHUMG00000000960.1,NA,ENSG00000237613,chr1,34553,37596
ENSG00000186092.7,HAVANA,gene,ENSG00000186092.7,protein_coding,OR4F5,2,NA,HGNC:14825,OTTHUMG00000001094.4,NA,ENSG00000186092,chr1,65419,71586


In [4]:
exprdata <- read.delim(expr.file, sep="\t", as.is=T, skip=2)
nrow(exprdata)
head(exprdata)

[1] 59033

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000223972.5,DDX11L1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.167751,0.00000,0.0000,0.00000,0.00000
2,ENSG00000227232.5,WASH7P,3.99789,3.17815,2.673080,4.0708,3.87547,3.62625,5.05094,1.459020,⋯,6.04259,3.041170,3.328970,2.873740,4.0856800,4.542470,6.31791,7.0687,5.74187,2.89503
3,ENSG00000278267.1,MIR6859-1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000
4,ENSG00000243485.5,MIR1302-2HG,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.054907,0.00000,0.0000,0.00000,0.00000
5,ENSG00000237613.2,FAM138A,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.00000,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000
6,ENSG00000268020.3,OR4G4P,0.00000,0.00000,0.035769,0.0000,0.00000,0.00000,0.00000,0.044141,⋯,0.00000,0.032181,0.038557,0.038184,0.0686565,0.000000,0.00000,0.0000,0.00000,0.00000


In [5]:
# add ensemlb id base (without decimal) for comparison to rowdata
exprdata$Name_base <- exprdata$Name %>%
    lapply(FUN=strsplit, split="\\.") %>%
    unlist() %>%
    matrix(ncol=2, byrow=T)  %>%
    subset.matrix(select = 1) %>%
    as.character()

# remove small number of duplicate ensembl id base vals
temp.dups <- exprdata$Name_base[which(duplicated(exprdata$Name_base))]
exprdata <- exprdata[which(!exprdata$Name_base %in% temp.dups),]
rownames(exprdata) <- exprdata$Name_base
nrow(exprdata)
head(exprdata)

[1] 58943

,Name,Description,Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,⋯,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood,Name_base
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
ENSG00000223972,ENSG00000223972.5,DDX11L1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.167751,0.00000,0.0000,0.00000,0.00000,ENSG00000223972
ENSG00000227232,ENSG00000227232.5,WASH7P,3.99789,3.17815,2.673080,4.0708,3.87547,3.62625,5.05094,1.459020,⋯,3.041170,3.328970,2.873740,4.0856800,4.542470,6.31791,7.0687,5.74187,2.89503,ENSG00000227232
ENSG00000278267,ENSG00000278267.1,MIR6859-1,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000278267
ENSG00000243485,ENSG00000243485.5,MIR1302-2HG,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.054907,0.00000,0.0000,0.00000,0.00000,ENSG00000243485
ENSG00000237613,ENSG00000237613.2,FAM138A,0.00000,0.00000,0.000000,0.0000,0.00000,0.00000,0.00000,0.000000,⋯,0.000000,0.000000,0.000000,0.0000000,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000237613
ENSG00000268020,ENSG00000268020.3,OR4G4P,0.00000,0.00000,0.035769,0.0000,0.00000,0.00000,0.00000,0.044141,⋯,0.032181,0.038557,0.038184,0.0686565,0.000000,0.00000,0.0000,0.00000,0.00000,ENSG00000268020


# Make coldata table

In [6]:
colnames(exprdata)[3:ncol(exprdata)-1]

[1] "Description"                                      
 [2] "Adipose_Subcutaneous"                             
 [3] "Adipose_Visceral_Omentum"                         
 [4] "Adrenal_Gland"                                    
 [5] "Artery_Aorta"                                     
 [6] "Artery_Coronary"                                  
 [7] "Artery_Tibial"                                    
 [8] "Bladder"                                          
 [9] "Brain_Amygdala"                                   
[10] "Brain_Anterior_cingulate_cortex_BA24"             
[11] "Brain_Caudate_basal_ganglia"                      
[12] "Brain_Cerebellar_Hemisphere"                      
[13] "Brain_Cerebellum"                                 
[14] "Brain_Cortex"                                     
[15] "Brain_Frontal_Cortex_BA9"                         
[16] "Brain_Hippocampus"                                
[17] "Brain_Hypothalamus"                               
[18] "Brain_Nucleus_accumbens_basal_ganglia"            
[19] "Brain_Putamen_basal_ganglia"                      
[20] "Brain_Spinal_cord_cervical_c.1"                   
[21] "Brain_Substantia_nigra"                           
[22] "Breast_Mammary_Tissue"                            
[23] "Cells_Cultured_fibroblasts"                       
[24] "Cells_EBV.transformed_lymphocytes"                
[25] "Cervix_Ectocervix"                                
[26] "Cervix_Endocervix"                                
[27] "Colon_Sigmoid"                                    
[28] "Colon_Transverse"                                 
[29] "Colon_Transverse_Mixed_Cell"                      
[30] "Colon_Transverse_Mucosa"                          
[31] "Colon_Transverse_Muscularis"                      
[32] "Esophagus_Gastroesophageal_Junction"              
[33] "Esophagus_Mucosa"                                 
[34] "Esophagus_Muscularis"                             
[35] "Fallopian_Tube"                                   
[36] "Heart_Atrial_Appendage"                           
[37] "Heart_Left_Ventricle"                             
[38] "Kidney_Cortex"                                    
[39] "Kidney_Medulla"                                   
[40] "Liver"                                            
[41] "Liver_Hepatocyte"                                 
[42] "Liver_Mixed_Cell"                                 
[43] "Liver_Portal_Tract"                               
[44] "Lung"                                             
[45] "Minor_Salivary_Gland"                             
[46] "Muscle_Skeletal"                                  
[47] "Nerve_Tibial"                                     
[48] "Ovary"                                            
[49] "Pancreas"                                         
[50] "Pancreas_Acini"                                   
[51] "Pancreas_Islets"                                  
[52] "Pancreas_Mixed_Cell"                              
[53] "Pituitary"                                        
[54] "Prostate"                                         
[55] "Skin_Not_Sun_Exposed_Suprapubic"                  
[56] "Skin_Sun_Exposed_Lower_leg"                       
[57] "Small_Intestine_Terminal_Ileum"                   
[58] "Small_Intestine_Terminal_Ileum_Lymphode_Aggregate"
[59] "Small_Intestine_Terminal_Ileum_Mixed_Cell"        
[60] "Spleen"                                           
[61] "Stomach"                                          
[62] "Stomach_Mixed_Cell"                               
[63] "Stomach_Mucosa"                                   
[64] "Stomach_Muscularis"                               
[65] "Testis"                                           
[66] "Thyroid"                                          
[67] "Uterus"                                           
[68] "Vagina"                                           
[69] "Whole_Blood"

In [7]:
samples <- colnames(exprdata)[3:(ncol(exprdata)-1)]
coldata <- cbind(samples)
colnames(coldata) <- c("label")
rownames(coldata) <- samples
nrow(coldata)
head(coldata)

[1] 68

,label
Adipose_Subcutaneous,Adipose_Subcutaneous
Adipose_Visceral_Omentum,Adipose_Visceral_Omentum
Adrenal_Gland,Adrenal_Gland
Artery_Aorta,Artery_Aorta
Artery_Coronary,Artery_Coronary
Artery_Tibial,Artery_Tibial


# Make counts table

In [8]:
#match rowdata with exprdata

m <- match(rowdata[,match.col], rownames(exprdata))

# check matches
length(m)
length(which(is.na(m))) # how many NAs?
all(rowdata[which(!is.na(m)),match.col] == rownames(exprdata)[m[which(!is.na(m))]]) # all non-NAs match correctly?

[1] 30696

[1] 994

[1] TRUE

In [9]:
# make exprmat
exprmat <- matrix(NA, nrow=nrow(rowdata), ncol=nrow(coldata))
rownames(exprmat) <- rownames(rowdata)
colnames(exprmat) <- rownames(coldata)
exprmat[,rownames(coldata)]  <- as.matrix(exprdata[m,coldata[rownames(coldata),"label"]])
exprmat <- apply(exprmat, 2, as.numeric)
nrow(exprmat)
head(exprmat)

[1] 30696

Adipose_Subcutaneous,Adipose_Visceral_Omentum,Adrenal_Gland,Artery_Aorta,Artery_Coronary,Artery_Tibial,Bladder,Brain_Amygdala,Brain_Anterior_cingulate_cortex_BA24,Brain_Caudate_basal_ganglia,⋯,Spleen,Stomach,Stomach_Mixed_Cell,Stomach_Mucosa,Stomach_Muscularis,Testis,Thyroid,Uterus,Vagina,Whole_Blood
NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0.000000,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000000,⋯,0.0000,0.000000,0.000000,0.000000,0.0000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000
0.000000,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0236910,⋯,0.0000,0.000000,0.000000,0.000000,0.0000000,0.0549070,0.0000000,0.000000,0.0000000,0.000000
NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
0.000000,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0000000,⋯,0.0000,0.000000,0.000000,0.000000,0.0000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000
0.040547,0.03639,0.05373,0.025192,0.03537,0.033259,0.026951,0.065919,0.059381,0.0625085,⋯,0.0275,0.050328,0.077067,0.067548,0.0686105,0.0336705,0.0395405,0.026106,0.0273745,0.017036


# write output

In [10]:
saveRDS(coldata, file=out.files[["coldata"]])
saveRDS(exprmat, file=out.files[["exprmat"]])
saveRDS(coldata, file=out.files[["layout"]])